# RMSE planimétrico contra la geodatabase IGAC

Validación posicional de los polígonos generados (Anexo E) contra la geodatabase de referencia del IGAC.

**Procedimiento:**
1. Cargar polígonos predichos (`results/vectores/construccion_<ciudad>.gpkg`).
2. Cargar polígonos de referencia desde `data/processed/vectores_consolidados_clase/<ciudad>_consolidated.gpkg` (clase `Construccion`).
3. Pareo **nearest-neighbor** entre centroides predichos y centroides de referencia (KDTree).
4. Distancia 2D por par.
5. Reportar **RMSE**, percentiles (P50, P90, **P95**, P99) y **LE95** (Linear Error 95%) — equivalente al percentil 95 de las distancias 2D, métrica estándar NSSDA/NMAS para precisión planimétrica.

Sólo se evalúa la clase **Construcción** (la única con polígonos discretos comparables uno-a-uno; Bosque, Vías y similares son entidades continuas).

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path(os.path.abspath('..')).resolve()

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

CIUDADES = ['madrid', 'mosquera', 'tabio', 'tocancipa']
CLASE_CONSTRUCCION = 1

# Filtros de polígonos pequeños (artefactos de rasterización/consolidación)
MIN_AREA_M2 = 50.0   # mismo umbral usado en el postprocesamiento RDP

# Distancia máxima de pareo (m) — descarta pares forzados que no corresponden
MAX_PAIRING_DIST_M = 50.0

print(f'ROOT: {ROOT}')
print(f'Ciudades: {CIUDADES}')
print(f'Filtros: area >= {MIN_AREA_M2} m² · max pairing distance = {MAX_PAIRING_DIST_M} m')

## 1. Funciones de carga

In [ ]:
def load_predicted(ciudad: str) -> gpd.GeoDataFrame:
    """Polígonos generados por el pipeline (Anexo E)."""
    path = ROOT / 'results' / 'vectores' / f'construccion_{ciudad}.gpkg'
    g = gpd.read_file(path)
    g = g[g.geometry.area >= MIN_AREA_M2].reset_index(drop=True)
    g.set_crs('EPSG:9377', allow_override=True, inplace=True)
    return g

def load_reference(ciudad: str) -> gpd.GeoDataFrame:
    """Polígonos de referencia IGAC (clase Construccion) ya consolidados."""
    path = ROOT / 'data' / 'processed' / 'vectores_consolidados_clase' / f'{ciudad}_consolidated.gpkg'
    g = gpd.read_file(path)
    g = g[g['class_id'] == CLASE_CONSTRUCCION]
    g = g.explode(index_parts=False).reset_index(drop=True)
    g = g[g.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])]
    g = g.explode(index_parts=False).reset_index(drop=True)
    g = g[g.geometry.area >= MIN_AREA_M2].reset_index(drop=True)
    g.set_crs('EPSG:9377', allow_override=True, inplace=True)
    return g

# Probar carga
test = load_reference('madrid')
print(f'Madrid · referencia: {len(test):,} polígonos · área media {test.geometry.area.mean():.1f} m²')
test = load_predicted('madrid')
print(f'Madrid · predichos:  {len(test):,} polígonos · área media {test.geometry.area.mean():.1f} m²')

## 2. Pareo nearest-neighbor + cálculo de distancias 2D

La función devuelve la distancia entre el centroide de cada polígono predicho y el centroide del polígono de referencia más cercano. Pares con distancia > `MAX_PAIRING_DIST_M` se descartan (no son correspondencias reales).

In [ ]:
def pair_centroids(pred: gpd.GeoDataFrame, ref: gpd.GeoDataFrame, max_dist: float):
    pred_xy = np.array([(p.x, p.y) for p in pred.geometry.centroid])
    ref_xy = np.array([(p.x, p.y) for p in ref.geometry.centroid])
    tree = cKDTree(ref_xy)
    dists, idx = tree.query(pred_xy, k=1)
    keep = dists <= max_dist
    return dists[keep], idx[keep], keep, pred_xy, ref_xy

def compute_metrics(distances: np.ndarray) -> dict:
    if len(distances) == 0:
        return {k: np.nan for k in ['n', 'rmse', 'mean', 'p50', 'p90', 'p95', 'p99', 'le95']}
    return {
        'n':     int(len(distances)),
        'rmse':  float(np.sqrt(np.mean(distances ** 2))),
        'mean':  float(np.mean(distances)),
        'p50':   float(np.percentile(distances, 50)),
        'p90':   float(np.percentile(distances, 90)),
        'p95':   float(np.percentile(distances, 95)),
        'p99':   float(np.percentile(distances, 99)),
        'le95':  float(np.percentile(distances, 95)),  # NSSDA: LE95 = P95 de error 2D
    }

## 3. Evaluar los 4 municipios

In [ ]:
results = {}
all_distances = {}
for ciudad in CIUDADES:
    print(f'\n=== {ciudad} ===')
    pred = load_predicted(ciudad)
    ref = load_reference(ciudad)
    print(f'  Predichos:   {len(pred):>5,} polígonos')
    print(f'  Referencia:  {len(ref):>5,} polígonos')
    if len(pred) == 0 or len(ref) == 0:
        print('  [skip] sin datos suficientes')
        continue
    distances, idx, keep, pred_xy, ref_xy = pair_centroids(pred, ref, MAX_PAIRING_DIST_M)
    pct_paired = len(distances) / len(pred) * 100
    print(f'  Pareados (<{MAX_PAIRING_DIST_M:.0f} m): {len(distances):>5,}  ({pct_paired:.1f}%)')
    metrics = compute_metrics(distances)
    metrics['n_predicted'] = len(pred)
    metrics['n_reference'] = len(ref)
    metrics['pct_paired'] = pct_paired
    results[ciudad] = metrics
    all_distances[ciudad] = distances
    print(f'  RMSE = {metrics["rmse"]:.2f} m   '
          f'Mediana = {metrics["p50"]:.2f} m   '
          f'P95 = {metrics["p95"]:.2f} m   '
          f'LE95 = {metrics["le95"]:.2f} m')

## 4. Tabla resumen y consolidado global

In [ ]:
# Distancias agregadas para reporte global
dist_global = np.concatenate([d for d in all_distances.values()]) if all_distances else np.array([])
metrics_global = compute_metrics(dist_global)
metrics_global['n_predicted'] = sum(r['n_predicted'] for r in results.values())
metrics_global['n_reference'] = sum(r['n_reference'] for r in results.values())
metrics_global['pct_paired'] = (
    metrics_global['n'] / metrics_global['n_predicted'] * 100
    if metrics_global['n_predicted'] else np.nan
)

df = pd.DataFrame(results).T
df.loc['GLOBAL'] = pd.Series(metrics_global)
df = df[['n_predicted', 'n_reference', 'n', 'pct_paired', 'rmse', 'mean', 'p50', 'p90', 'p95', 'p99', 'le95']]
df.columns = ['n_pred', 'n_ref', 'n_pareados', '% pareo', 'RMSE (m)', 'Media (m)', 'P50 (m)', 'P90 (m)', 'P95 (m)', 'P99 (m)', 'LE95 (m)']
df.round(2)

In [ ]:
out_csv = ROOT / 'results' / 'comparativa_tres_modelos' / 'rmse_planimetrico.csv'
out_csv.parent.mkdir(parents=True, exist_ok=True)
df.round(3).to_csv(out_csv)
print(f'Tabla guardada: {out_csv}')

## 5. Histograma de distancias 2D

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, ciudad in zip(axes.flatten(), CIUDADES):
    if ciudad not in all_distances:
        continue
    d = all_distances[ciudad]
    m = results[ciudad]
    ax.hist(d, bins=40, color='#2E86C1', edgecolor='black', linewidth=0.4, alpha=0.8)
    ax.axvline(m['rmse'], color='#E74C3C', linestyle='-', lw=2,
               label=f"RMSE = {m['rmse']:.2f} m")
    ax.axvline(m['p50'], color='#27AE60', linestyle='--', lw=2,
               label=f"Mediana = {m['p50']:.2f} m")
    ax.axvline(m['le95'], color='#8E44AD', linestyle='--', lw=2,
               label=f"LE95 = {m['le95']:.2f} m")
    ax.set_xlabel('Distancia 2D entre centroides (m)')
    ax.set_ylabel('Frecuencia')
    ax.set_title(f'{ciudad.capitalize()} (n = {m["n"]:,})', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
fig.suptitle('Error planimétrico — pareo nearest-neighbor entre centroides predichos y referencia IGAC',
             fontsize=13, fontweight='bold', y=1.005)
plt.tight_layout()
out_fig = ROOT / 'results' / 'comparativa_tres_modelos' / 'fig_rmse_histogramas.png'
plt.savefig(out_fig, dpi=200, bbox_inches='tight')
plt.show()
print(f'Figura guardada: {out_fig}')

In [ ]:
# Histograma global
fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(dist_global, bins=60, color='#2E86C1', edgecolor='black', linewidth=0.4, alpha=0.8)
ax.axvline(metrics_global['rmse'], color='#E74C3C', linestyle='-', lw=2.5,
           label=f"RMSE global = {metrics_global['rmse']:.2f} m")
ax.axvline(metrics_global['p50'], color='#27AE60', linestyle='--', lw=2,
           label=f"Mediana = {metrics_global['p50']:.2f} m")
ax.axvline(metrics_global['le95'], color='#8E44AD', linestyle='--', lw=2,
           label=f"LE95 = {metrics_global['le95']:.2f} m")
ax.set_xlabel('Distancia 2D entre centroides (m)')
ax.set_ylabel('Frecuencia')
ax.set_title(f'Error planimétrico GLOBAL — los 4 municipios (n = {metrics_global["n"]:,} pares)',
             fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
out_fig = ROOT / 'results' / 'comparativa_tres_modelos' / 'fig_rmse_global.png'
plt.savefig(out_fig, dpi=200, bbox_inches='tight')
plt.show()

## 6. Interpretación según estándares cartográficos

Para escala 1:1 000 los estándares de exactitud posicional típicos son:

- **NSSDA / NMAS:** RMSE planimétrico ≤ 0.5 m (escala) → LE95 ≤ 0.866 m
- **IGAC para cartografía base 1:1 000:** tolerancia típica ≤ 0.5 m
- **OGC ISO 19157:** P95 reportado como métrica de exactitud absoluta posicional

El valor LE95 reportado arriba indica que **el 95% de los polígonos predichos** tienen su centroide a una distancia menor o igual a ese valor del centroide del polígono de referencia más cercano.

In [ ]:
thresholds = [0.5, 0.866, 1.0, 2.0, 5.0]
print('Porcentaje de polígonos predichos con error ≤ X m (global):')
print()
for t in thresholds:
    pct = (dist_global <= t).sum() / len(dist_global) * 100
    print(f'  ≤ {t:>5.2f} m  →  {pct:5.2f}%')